In [1]:
# ------------------------------------------------------------
# 0. Librerías
# ------------------------------------------------------------
#pip install nltk datasets pandas -q

import re
import random
import unicodedata
import pandas as pd
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
from nltk.classify import NaiveBayesClassifier, accuracy
from datasets import load_dataset

# Lista de palabras conectoras en inglés que se usa en el preprocesamiento
nltk.download("stopwords", quiet=True)

# Semilla para que la elección de respuestas sea la misma en cada ejecución
random.seed(42)

c:\Users\marte\Documents\GitHub\Desarrollo_Aplicacion_IA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ------------------------------------------------------------
# 1. Configuración inicial de herramientas NLP
# ------------------------------------------------------------

# Separa el texto en palabras y reduce cada palabra a su raíz (cancel, canceling, cancelled -> cancel)
tokenizer = RegexpTokenizer(r'\w+')
stemmer = SnowballStemmer("english")

# Palabras conectoras que no aportan a la intención (the, my, you, to...)
# Se dejan las negaciones porque cambian el sentido del mensaje
STOPWORDS = set(stopwords.words("english")) - {"no", "not", "nor"}

def normalizar_texto(texto):
    """
    Convierte el texto a minúsculas, elimina acentos y caracteres especiales.
    """
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^a-zñ0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def tokenizar_y_stem(texto):
    """
    Tokeniza el texto, quita las palabras conectoras y aplica stemming.
    """
    texto = normalizar_texto(texto)
    tokens = [token for token in tokenizer.tokenize(texto) if token not in STOPWORDS]
    stems = [stemmer.stem(token) for token in tokens]
    return tokens, stems

In [3]:
# ------------------------------------------------------------
# 2. Carga, limpieza y división del Bitext Customer Support Dataset
# ------------------------------------------------------------

# Cargando base de datos desde Hugging Face
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = ds["train"].to_pandas()

# Limpieza de valores faltantes y espacios sobrantes
df = df.dropna(subset=["instruction", "intent", "response"])
for col in ["instruction", "category", "intent", "response", "flags"]:
    df[col] = df[col].astype(str).str.strip()

# Una misma consulta puede repetirse con respuestas distintas,
# así que se conserva una vez cada par consulta-intención
df = df.drop_duplicates(subset=["instruction", "intent"]).reset_index(drop=True)

# Split 80/20 por intención: el 80 % se usa para entrenar el modelo y armar las
# intenciones, y el 20 % para probar con mensajes que el chatbot no conoce.
# random_state fija la división para que siempre salga igual
PORCENTAJE_PRUEBA = 0.2
prueba = df.groupby("intent", group_keys=False).sample(frac=PORCENTAJE_PRUEBA, random_state=42)
entrenamiento = df.drop(prueba.index)

print(f"Categorías: {df['category'].nunique()} | Intenciones: {df['intent'].nunique()}")

Categorías: 11 | Intenciones: 27


In [4]:
# ------------------------------------------------------------
# 2.1 Base de conocimiento: datos de la tienda
# ------------------------------------------------------------
# Datos de la tienda que se insertan en las respuestas con .format()

DATOS_TIENDA = {
    "website": "www.ourstore.com",
    "support_email": "support@ourstore.com",
    "support_phone": "1-800-555-0123",
    "support_hours": "Monday to Friday, 9 a.m. to 6 p.m. (CT)",
    "standard_shipping": "5 to 7 business days",
    "express_shipping": "2 to 3 business days",
    "free_shipping_min": "$75",
    "return_window": "30 days",
    "refund_time": "5 to 10 business days",
    "cancellation_window": "before your order ships",
    "cancellation_fee": "no fee",
}

In [5]:
# ------------------------------------------------------------
# 2.2 Base de conocimiento: respuestas por intención
# ------------------------------------------------------------
# Intenciones que no resuelve el chatbot y se canalizan a un agente humano
INTENCIONES_SENSIBLES = {"complaint", "payment_issue", "contact_human_agent", "delete_account"}

# Dos respuestas por intención; los marcadores {website}, {refund_time}, etc.
# se reemplazan con DATOS_TIENDA al momento de responder
RESPUESTAS = {
    # ACCOUNT
    "create_account": [
        "Happy to help you get started! Just go to {website}, click on 'Sign up' and fill in your name, email and a password. It only takes a minute.",
        "Creating an account is quick and easy. Head over to {website}, choose 'Sign up' and follow the steps. Once you're in, you'll be able to track your orders and save your addresses.",
    ],
    "delete_account": [
        "I'm sorry to see you go. To make sure everything is handled safely, I'll connect you with a member of our team who can help you close your account.",
        "Thanks for letting us know. Account deletion needs to be confirmed by our support team, so I'm passing your request to an agent who will reach out shortly.",
    ],
    "edit_account": [
        "Sure thing! Sign in at {website}, go to 'My account' and click 'Edit profile' to update your details. Don't forget to save your changes.",
        "You can update your name, email or phone number anytime from 'My account' at {website}. Just click 'Edit profile' and save when you're done.",
    ],
    "recover_password": [
        "No worries, it happens! Go to {website}, click 'Sign in' and then 'Forgot password'. We'll send you an email with a link to create a new one.",
        "Let's get you back in. Click 'Forgot password' on the sign-in page at {website} and check your inbox for the reset link. If you don't see it, take a look at your spam folder.",
    ],
    "registration_problems": [
        "Sorry you're having trouble signing up. Please make sure your email isn't already registered and that your password has at least 8 characters. If it still doesn't work, write to us at {support_email} and we'll sort it out.",
        "That's frustrating, let's fix it. Try refreshing the page or using a different browser. If the problem continues, our team can help you at {support_email}.",
    ],
    "switch_account": [
        "To switch accounts, just sign out from 'My account' at {website} and sign back in with the other email.",
        "Sure! Sign out of your current account and then sign in with the one you'd like to use. Your orders and addresses stay saved in each account.",
    ],
    # CANCEL
    "check_cancellation_fee": [
        "Good news, there's {cancellation_fee} for canceling an order as long as you do it {cancellation_window}.",
        "Canceling an order has {cancellation_fee} if it's done {cancellation_window}. If it has already shipped, you can return it within {return_window} instead.",
    ],
    # CONTACT
    "contact_customer_service": [
        "You can reach our customer service team at {support_email} or call us at {support_phone}, {support_hours}.",
        "We're happy to help! Write to us at {support_email} or give us a call at {support_phone}. Our hours are {support_hours}.",
    ],
    "contact_human_agent": [
        "Of course! I'm connecting you with one of our agents right now. They'll be able to see our conversation, so you won't have to repeat anything.",
        "Sure, let me pass you to a member of our team. Please hold on for a moment while an agent joins the chat.",
    ],
    # DELIVERY
    "delivery_options": [
        "We offer standard shipping ({standard_shipping}) and express shipping ({express_shipping}). Standard shipping is free on orders over {free_shipping_min}.",
        "You can choose between standard delivery, which takes {standard_shipping}, and express delivery, which takes {express_shipping}. You'll see both options at checkout.",
    ],
    "delivery_period": [
        "Standard orders usually arrive in {standard_shipping}, and express orders in {express_shipping}. You'll get a tracking link by email as soon as your order ships.",
        "Delivery times depend on the option you chose: {standard_shipping} for standard shipping and {express_shipping} for express. You can check the status anytime under 'My orders'.",
    ],
    # FEEDBACK
    "complaint": [
        "I'm really sorry about your experience. I want to make sure this gets the attention it deserves, so I'm connecting you with a member of our team.",
        "Thanks for telling us, and I'm sorry things didn't go as expected. An agent will join the chat shortly to help you resolve it.",
    ],
    "review": [
        "We'd love to hear your thoughts! You can leave a review on the product page at {website}, under 'Write a review'.",
        "Thanks for wanting to share your opinion! Just go to the product page, scroll down to 'Reviews' and tell us what you think.",
    ],
    # INVOICE
    "check_invoice": [
        "You can find all your invoices in 'My account' > 'My orders' at {website}. Just select the order you need.",
        "Sure! Sign in at {website}, go to 'My orders' and click on the order to see its invoice.",
    ],
    "get_invoice": [
        "To download your invoice, go to 'My orders' at {website}, open the order and click 'Download invoice'.",
        "No problem! Your invoice is available as a PDF in 'My orders'. If you can't find it, write to us at {support_email} and we'll send it to you.",
    ],
    # SUBSCRIPTION
    "newsletter_subscription": [
        "Want to hear about new arrivals and special offers? Just enter your email at the bottom of {website} and click 'Subscribe'.",
        "You can subscribe or unsubscribe anytime. To join, add your email at the bottom of our page, and to leave, click 'Unsubscribe' in any of our emails.",
    ],
    # ORDER
    "cancel_order": [
        "You can cancel your order {cancellation_window}. Go to 'My orders' at {website}, select the order and click 'Cancel order'.",
        "Sure, I can help with that. Open 'My orders', choose the order you want to cancel and click 'Cancel order'. If it has already shipped, you can return it within {return_window}.",
    ],
    "change_order": [
        "If your order hasn't shipped yet, you can change it from 'My orders' at {website}. Just open the order and click 'Edit order'.",
        "Happy to help! Changes like size, color or quantity can be made {cancellation_window} from 'My orders'. If it's already on its way, you can return or exchange it once it arrives.",
    ],
    "place_order": [
        "Placing an order is easy! Add the items you like to your cart at {website}, click 'Checkout' and follow the steps to choose your shipping and payment method.",
        "Just pick your items, choose your size and add them to your cart. When you're ready, click 'Checkout' to complete your purchase.",
    ],
    "track_order": [
        "You can track your order from 'My orders' at {website}. We also sent you a tracking link by email when your order shipped.",
        "Sure! Sign in at {website} and go to 'My orders' to see where your package is. Standard orders usually arrive in {standard_shipping}.",
    ],
    # PAYMENT
    "check_payment_methods": [
        "We accept Visa, Mastercard, American Express, PayPal and Apple Pay.",
        "You can pay with any major credit or debit card, PayPal or Apple Pay. You'll see all the options at checkout.",
    ],
    "payment_issue": [
        "I'm sorry you're having trouble with your payment. Since this involves your billing details, I'm connecting you with an agent who can review it safely.",
        "Thanks for letting us know. Payment issues need to be checked by our team, so I'm passing you to an agent who will help you right away.",
    ],
    # REFUND
    "check_refund_policy": [
        "You can return any item within {return_window} of delivery for a full refund, as long as it's unworn and has its tags.",
        "Our refund policy is simple: items can be returned within {return_window} in their original condition, and the refund goes back to your original payment method.",
    ],
    "get_refund": [
        "To request a refund, go to 'My orders' at {website}, select the order and click 'Return item'. Once we receive it, your refund will be processed in {refund_time}.",
        "Sure! Start a return from 'My orders' and send the item back within {return_window}. We'll issue your refund as soon as it reaches us.",
    ],
    "track_refund": [
        "Refunds usually take {refund_time} to show up after we receive your return. You can check the status in 'My orders' at {website}.",
        "Your refund is processed once your return arrives at our warehouse, and it may take {refund_time} to appear in your account. You can follow its status in 'My orders'.",
    ],
    # SHIPPING
    "change_shipping_address": [
        "If your order hasn't shipped yet, you can update the address from 'My orders' at {website}. For future orders, you can change it in 'My account' > 'Addresses'.",
        "Sure! Go to 'My account' > 'Addresses' to update it. If you need to change the address of an order that's already on its way, please contact us at {support_email}.",
    ],
    "set_up_shipping_address": [
        "To add a shipping address, sign in at {website}, go to 'My account' > 'Addresses' and click 'Add new address'.",
        "You can save one or more addresses in 'My account' > 'Addresses', so they're ready to choose at checkout.",
    ],
}

In [6]:
# ------------------------------------------------------------
# 2.3 Diccionario de intenciones
# ------------------------------------------------------------
# Cada intención guarda su categoría

intenciones = {}
for intent, grupo in entrenamiento.groupby("intent"):
    intenciones[intent] = {
        "categoria": grupo["category"].iloc[0],
        "accion": "canalizar_agente" if intent in INTENCIONES_SENSIBLES else "respuesta_automatica",
        "ejemplos": grupo["instruction"].tolist(),
        "respuestas": RESPUESTAS.get(intent, []),
    }

# Resumen de las intenciones definidas
resumen_intenciones = pd.DataFrame([
    {"categoria": d["categoria"], "intencion": i, "ejemplos": len(d["ejemplos"]),
     "respuestas": len(d["respuestas"]), "accion": d["accion"]}
    for i, d in intenciones.items()
]).sort_values(["categoria", "intencion"]).reset_index(drop=True)

resumen_intenciones

,categoria,intencion,ejemplos,respuestas,accion
0,ACCOUNT,create_account,714,2,respuesta_automatica
1,ACCOUNT,delete_account,734,2,canalizar_agente
2,ACCOUNT,edit_account,622,2,respuesta_automatica
3,ACCOUNT,recover_password,796,2,respuesta_automatica
4,ACCOUNT,registration_problems,799,2,respuesta_automatica
5,ACCOUNT,switch_account,690,2,respuesta_automatica
6,CANCEL,check_cancellation_fee,760,2,respuesta_automatica
7,CONTACT,contact_customer_service,800,2,respuesta_automatica
8,CONTACT,contact_human_agent,799,2,canalizar_agente
9,DELIVERY,delivery_options,528,2,respuesta_automatica


In [7]:
# ------------------------------------------------------------
# 3. Entrenamiento del modelo Naive Bayes
# ------------------------------------------------------------

def extraer_caracteristicas(texto):
    """
    Convierte el texto en características para el modelo (un stem = una característica).
    """
    _, stems = tokenizar_y_stem(texto)
    return {stem: True for stem in stems}

# Cada frase se convierte en (características, intención)
datos_entrenamiento = [(extraer_caracteristicas(texto), intent)
                       for texto, intent in zip(entrenamiento["instruction"], entrenamiento["intent"])]
datos_prueba = [(extraer_caracteristicas(texto), intent)
                for texto, intent in zip(prueba["instruction"], prueba["intent"])]

# El modelo aprende con el 80 % y se mide con el 20 % que no vio
modelo_nb = NaiveBayesClassifier.train(datos_entrenamiento)

print(f"Precisión del modelo en prueba: {accuracy(modelo_nb, datos_prueba):.2%}\n")


Precisión del modelo en prueba: 98.13%



In [8]:
# ------------------------------------------------------------
# 4. Detección de intención
# ------------------------------------------------------------

# Regla: probabilidad mínima para aceptar la intención que predice el modelo
UMBRAL = 0.4

def detectar_intencion(mensaje):
    """
    Detecta la intención con el modelo Naive Bayes. Devuelve intención y probabilidad.
    """
    caracteristicas = extraer_caracteristicas(mensaje)

    # Si después del preprocesamiento no queda ninguna palabra, no hay nada que clasificar
    if not caracteristicas:
        return "desconocida", 0

    # El modelo calcula la probabilidad de cada intención y se toma la más alta
    probabilidades = modelo_nb.prob_classify(caracteristicas)
    mejor_intencion = probabilidades.max()
    probabilidad = probabilidades.prob(mejor_intencion)

    # Si el modelo no está seguro se toma como fuera del dominio
    if probabilidad < UMBRAL:
        return "desconocida", round(probabilidad, 2)

    return mejor_intencion, round(probabilidad, 2)

In [9]:
# ------------------------------------------------------------
# 5. Detección básica de entidades
# ------------------------------------------------------------
# Catálogo de palabras clave por entidad: valor normalizado -> palabras que lo representan

catalogo_entidades = {
    "prenda": {
        "jacket": ["jacket", "coat", "hoodie", "sweater"],
        "shirt": ["shirt", "t-shirt", "tshirt", "blouse", "top"],
        "pants": ["pants", "jeans", "trousers", "shorts", "leggings"],
        "dress": ["dress", "skirt"],
        "shoes": ["shoes", "sneakers", "boots", "sandals"],
        "accessories": ["hat", "cap", "scarf", "belt", "bag"]
    },
    "talla": {
        "XS": ["xs", "extra small"],
        "S": ["small"],
        "M": ["medium"],
        "L": ["large"],
        "XL": ["xl", "extra large"]
    },
    "color": {
        "black": ["black"], "white": ["white"], "blue": ["blue", "navy"],
        "red": ["red"], "green": ["green"], "gray": ["gray", "grey"]
    },
    "metodo_pago": {
        "credit card": ["credit card", "debit card", "visa", "mastercard", "amex"],
        "paypal": ["paypal"],
        "apple pay": ["apple pay"]
    },
    "tipo_envio": {
        "standard": ["standard", "regular"],
        "express": ["express", "fast", "overnight"]
    }
}

def detectar_entidades(mensaje):
    """
    Detecta entidades mediante diccionario de términos y el número de pedido con regex.
    """
    texto = normalizar_texto(mensaje)
    entidades = {}

    # Se busca cada palabra clave como palabra completa y se guarda el primer valor encontrado
    for entidad, valores in catalogo_entidades.items():
        for valor, palabras_clave in valores.items():
            for palabra in palabras_clave:
                # \b evita coincidencias dentro de otras palabras ("red" en "registered")
                if re.search(rf"\b{re.escape(palabra)}\b", texto):
                    entidades[entidad] = valor
                    break
            if entidad in entidades:
                break

    # Número de pedido: "order 12345" o "#12345" (4 dígitos o más)
    pedido = re.search(r"(?:order|#)\s*#?\s*(\d{4,})", texto)
    if pedido:
        entidades["numero_pedido"] = pedido.group(1)

    return entidades

In [10]:
# ------------------------------------------------------------
# 6. Generación de respuesta
# ------------------------------------------------------------

MENSAJE_FALLBACK = (
    "Sorry, I didn't catch that. Could you tell me what you need help with? "
    "I can help you with your orders, shipping, returns and refunds, payments or your account."
)

def generar_respuesta(mensaje):
    """
    Genera una respuesta considerando intención, reglas y entidades.
    """
    # Paso 1: intención con el modelo y entidades con el catálogo
    intencion, puntaje = detectar_intencion(mensaje)
    entidades = detectar_entidades(mensaje)

    # Paso 2: si no se reconoce la intención se pide al cliente reformular
    if intencion == "desconocida":
        return {
            "intencion": intencion,
            "puntaje": puntaje,
            "accion": "fallback",
            "entidades": entidades,
            "respuesta": MENSAJE_FALLBACK
        }

    # Paso 3: la regla de la intención define si responde el bot o un agente
    accion = intenciones[intencion]["accion"]

    # Paso 4: se elige una de las respuestas de la intención
    plantilla = random.choice(intenciones[intencion]["respuestas"])

    # Paso 5: se rellenan los marcadores con los datos de la tienda y las entidades
    valores = {**DATOS_TIENDA, "prenda": "item", **entidades}
    respuesta = plantilla.format(**valores)

    return {
        "intencion": intencion,
        "puntaje": puntaje,
        "accion": accion,
        "entidades": entidades,
        "respuesta": respuesta
    }

In [11]:
# ------------------------------------------------------------
# 7. Chatbot interactivo
# ------------------------------------------------------------

SALUDO = [
    "Hi there! I'm the virtual assistant for our online clothing store.",
    "I can help you with your orders, shipping, returns and refunds, payments or your account.",
    "Type 'exit' anytime to end the chat."
]
DESPEDIDA = "Thanks for chatting with us! Have a great day"

def iniciar_chatbot():
    """
    Ejecuta el chatbot en modo interactivo y devuelve el historial de la conversación.
    """
    historial = []

    for linea in SALUDO:
        print(f"Chatbot: {linea}")
    print()

    while True:
        # En VS Code la caja para escribir aparece en la parte superior de la ventana
        mensaje_usuario = input("Usuario (escribe arriba y presiona Enter): ").strip()

        # Se muestra el mensaje en la salida para que se vea la conversación completa
        print(f"Usuario: {mensaje_usuario}")

        # Si no se escribe nada, se vuelve a pedir el mensaje
        if not mensaje_usuario:
            print("Chatbot: Please type a message, or 'exit' to end the chat.\n")
            continue

        # El cliente termina la conversación con exit o salir
        if normalizar_texto(mensaje_usuario) in ["exit", "salir"]:
            print(f"Chatbot: {DESPEDIDA}")
            break

        resultado = generar_respuesta(mensaje_usuario)

        # Se guarda cada turno para el análisis posterior
        historial.append({
            "mensaje_usuario": mensaje_usuario,
            "intencion_detectada": resultado["intencion"],
            "puntaje": resultado["puntaje"],
            "accion": resultado["accion"],
            "entidades": resultado["entidades"],
            "respuesta": resultado["respuesta"]
        })

        print(f"Chatbot: {resultado['respuesta']}\n")

    return pd.DataFrame(historial)


In [14]:
historial = iniciar_chatbot()

Chatbot: Hi there! I'm the virtual assistant for our online clothing store.
Chatbot: I can help you with your orders, shipping, returns and refunds, payments or your account.
Chatbot: Type 'exit' anytime to end the chat.

Usuario: I want to return my dress I bought
Chatbot: You can cancel your order before your order ships. Go to 'My orders' at www.ourstore.com, select the order and click 'Cancel order'.

Usuario: Could you help me with my refund?
Chatbot: You can return any item within 30 days of delivery for a full refund, as long as it's unworn and has its tags.

Usuario: exit
Chatbot: Thanks for chatting with us! Have a great day


In [15]:
# ============================================================
# 8. Evaluación del chatbot
# ============================================================

# ------------------------------------------------------------
# 8.1 Evaluación con las frases de prueba del split
# ------------------------------------------------------------
# Se usa la instruction como mensaje del cliente y se compara con la intent real

resultados = []

for _, fila in prueba.iterrows():
    intencion, puntaje = detectar_intencion(fila["instruction"])
    resultados.append({
        "mensaje": fila["instruction"],
        "categoria": fila["category"],
        "intencion_esperada": fila["intent"],
        "intencion_detectada": intencion,
        "puntaje": puntaje,
        "correcto": fila["intent"] == intencion
    })

df_resultados = pd.DataFrame(resultados)

precision = df_resultados["correcto"].mean()
tasa_fallback = (df_resultados["intencion_detectada"] == "desconocida").mean()

print("Resultados de evaluación")
print("------------------------")
print(f"Frases de prueba: {len(df_resultados)}")
print(f"Precisión de intención: {precision:.2%}")
print(f"Tasa de fallback: {tasa_fallback:.2%}")

# Precisión por categoría
print("\nPrecisión por categoría")
print("------------------------")
print(df_resultados.groupby("categoria")["correcto"].mean()
      .sort_values().map("{:.2%}".format).to_string())

# Intenciones con menor precisión
print("\nIntenciones con menor precisión")
print("------------------------")
print(df_resultados.groupby("intencion_esperada")["correcto"].mean()
      .sort_values().head(8).map("{:.2%}".format).to_string())

# Los casos sensibles deben detectarse como sensibles para pasar a un agente
sensibles = df_resultados[df_resultados["intencion_esperada"].isin(INTENCIONES_SENSIBLES)]
canalizados = sensibles["intencion_detectada"].isin(INTENCIONES_SENSIBLES).mean()
print(f"\nCasos sensibles que se canalizarían a un agente: {canalizados:.2%}")

Resultados de evaluación
------------------------
Frases de prueba: 4927
Precisión de intención: 98.03%
Tasa de fallback: 0.43%

Precisión por categoría
------------------------
categoria
ORDER            93.53%
REFUND           95.42%
CONTACT          97.75%
DELIVERY         98.80%
INVOICE          98.91%
ACCOUNT          99.17%
CANCEL           99.47%
SHIPPING         99.49%
SUBSCRIPTION     99.50%
PAYMENT          99.50%
FEEDBACK        100.00%

Intenciones con menor precisión
------------------------
intencion_esperada
place_order            85.00%
get_refund             89.13%
cancel_order           92.93%
contact_human_agent    96.00%
delivery_options       97.73%
create_account         97.75%
change_order           98.28%
check_invoice          98.37%

Casos sensibles que se canalizarían a un agente: 98.98%


In [16]:
# ------------------------------------------------------------
# 8.2 Pruebas funcionales: diez conversaciones
# ------------------------------------------------------------
# Cada conversación simula a un cliente distinto con varios mensajes seguidos.
# Se indica la intención esperada de cada mensaje para verificar el comportamiento.

conversaciones = {
    "C01 Pedido y cancelación": [
        ("I want to place an order", "place_order"),
        ("actually, how can I cancel order 48213?", "cancel_order"),
    ],
    "C02 Opciones y tiempos de envío": [
        ("what shipping options do you have?", "delivery_options"),
        ("how long does express delivery take", "delivery_period"),
    ],
    "C03 Reembolso de una prenda": [
        ("what is your refund policy?", "check_refund_policy"),
        ("I want a refund for my blue dress", "get_refund"),
        ("how can I track my refund?", "track_refund"),
    ],
    "C04 Cuenta de usuario": [
        ("I'd like to create an account", "create_account"),
        ("I forgot my password", "recover_password"),
        ("how do I update my profile?", "edit_account"),
    ],
    "C05 Pagos": [
        ("which payment methods do you accept?", "check_payment_methods"),
        ("my paypal payment didn't go through", "payment_issue"),
    ],
    "C06 Dirección de envío": [
        ("I need to change my shipping address", "change_shipping_address"),
        ("how do I add a new shipping address?", "set_up_shipping_address"),
    ],
    "C07 Facturas": [
        ("where can I check my invoice?", "check_invoice"),
        ("I need to download the invoice for order 55120", "get_invoice"),
    ],
    "C08 Queja y agente humano": [
        ("I'm really upset, the shirt I got is the wrong size and nobody helps me", "complaint"),
        ("can I talk to a real person please", "contact_human_agent"),
    ],
    "C09 Contacto, boletín y opinión": [
        ("how can I contact customer service?", "contact_customer_service"),
        ("I want to subscribe to your newsletter", "newsletter_subscription"),
        ("I'd like to leave a review for the sneakers I bought", "review"),
    ],
    "C10 Fuera de dominio y cuenta": [
        ("do you sell tacos & pizza?", "desconocida"),
        ("is there a fee if I cancel my order?", "check_cancellation_fee"),
        ("please delete my account", "delete_account"),
    ],
}

def simular_conversacion(nombre, mensajes):
    """
    Ejecuta una conversación completa sin input() e imprime la transcripción.
    """
    print(f"===== {nombre} =====")
    for linea in SALUDO:
        print(f"Chatbot: {linea}")

    turnos = []
    for mensaje, esperada in mensajes:
        resultado = generar_respuesta(mensaje)
        print(f"\nUsuario: {mensaje}")
        print(f"Chatbot: {resultado['respuesta']}")
        print(f"  [intención: {resultado['intencion']} | probabilidad: {resultado['puntaje']} "
              f"| acción: {resultado['accion']} | entidades: {resultado['entidades']}]")
        turnos.append({
            "conversacion": nombre,
            "mensaje_usuario": mensaje,
            "intencion_esperada": esperada,
            "intencion_detectada": resultado["intencion"],
            "puntaje": resultado["puntaje"],
            "accion": resultado["accion"],
            "entidades": resultado["entidades"],
            "respuesta": resultado["respuesta"],
            "correcto": esperada == resultado["intencion"]
        })

    print("\nUsuario: exit")
    print(f"Chatbot: {DESPEDIDA}\n")
    return turnos

evidencias = []
for nombre, mensajes in conversaciones.items():
    evidencias.extend(simular_conversacion(nombre, mensajes))

df_evidencias = pd.DataFrame(evidencias)

# Se guardan las evidencias de ejecución en un archivo
df_evidencias.to_csv("evidencias_conversaciones.csv", index=False)

===== C01 Pedido y cancelación =====
Chatbot: Hi there! I'm the virtual assistant for our online clothing store.
Chatbot: I can help you with your orders, shipping, returns and refunds, payments or your account.
Chatbot: Type 'exit' anytime to end the chat.

Usuario: I want to place an order
Chatbot: Sorry, I didn't catch that. Could you tell me what you need help with? I can help you with your orders, shipping, returns and refunds, payments or your account.
  [intención: desconocida | probabilidad: 0.32 | acción: fallback | entidades: {}]

Usuario: actually, how can I cancel order 48213?
Chatbot: You can cancel your order before your order ships. Go to 'My orders' at www.ourstore.com, select the order and click 'Cancel order'.
  [intención: cancel_order | probabilidad: 1.0 | acción: respuesta_automatica | entidades: {'numero_pedido': '48213'}]

Usuario: exit
Chatbot: Thanks for chatting with us! Have a great day

===== C02 Opciones y tiempos de envío =====
Chatbot: Hi there! I'm the v

In [17]:
# ------------------------------------------------------------
# 8.3 Resumen de las pruebas funcionales
# ------------------------------------------------------------

print(f"Conversaciones: {df_evidencias['conversacion'].nunique()}")
print(f"Mensajes evaluados: {len(df_evidencias)}")
print(f"Intenciones correctas: {df_evidencias['correcto'].sum()} de {len(df_evidencias)} "
      f"({df_evidencias['correcto'].mean():.2%})")
print(f"Mensajes canalizados a un agente: {(df_evidencias['accion'] == 'canalizar_agente').sum()}")
print(f"Mensajes con fallback: {(df_evidencias['accion'] == 'fallback').sum()}")

# Resultado por conversación
(df_evidencias.groupby("conversacion")
 .agg(mensajes=("correcto", "size"), correctos=("correcto", "sum"))
 .reset_index())

Conversaciones: 10
Mensajes evaluados: 24
Intenciones correctas: 20 de 24 (83.33%)
Mensajes canalizados a un agente: 3
Mensajes con fallback: 2


,conversacion,mensajes,correctos
0,C01 Pedido y cancelación,2,1
1,C02 Opciones y tiempos de envío,2,2
2,C03 Reembolso de una prenda,3,2
3,C04 Cuenta de usuario,3,3
4,C05 Pagos,2,2
5,C06 Dirección de envío,2,2
6,C07 Facturas,2,2
7,C08 Queja y agente humano,2,1
8,"C09 Contacto, boletín y opinión",3,3
9,C10 Fuera de dominio y cuenta,3,2
